# Como o dashboard calcula cada card

Este notebook reimplementa, célula a célula, o cálculo de **cada card**
mostrado em `output/pll_metrics.html` — os valores não são copiados do
relatório, são recalculados aqui a partir dos CSVs brutos exportados pelo
MATLAB, com a fórmula ao lado de cada resultado.

**Cenários usados** (arbitrários, mas os dois com PLL **bem sintonizado** —
sem `_bad_pll`):

| Cenário | Pasta | Tipo |
|---|---|---|
| Regime permanente | `output/results/regime/` | sem falta |
| Curto trifásico na Barra 7 | `output/results/bus7/3phase/` | falta simétrica, `t_fault=0.30 s`, `t_clear=0.40 s` |

**Abordagem**: cada seção reimplementa a fórmula manualmente em NumPy/Pandas
(não chama o pipeline de produção para o cálculo em si), com o arquivo e a
linha do código-fonte citados em markdown. Para não deixar a cópia divergir
do código real com o tempo, toda seção termina com uma célula de
**verificação cruzada** — importa a classe de produção
(`SimData`/`SpectrumBuilder`) só para esse cenário e confere que o valor
manual bate com o valor real (`assert`). Se algum dia a fórmula de produção
mudar sem este notebook acompanhar, a célula de verificação **quebra**.

## Regra de manutenção

> Sempre que um card for **adicionado ou removido** em
> `src/report/renderer.py::_cards_html`, a seção correspondente deste
> notebook deve ser **adicionada ou removida junto** — ver a seção final
> "Cards atuais e política de sincronismo" para a lista completa hoje.
>
> **2026-08-09** — o grupo inteiro "Desempenho do PLL" (IAE, ISE, tₛ,
> |θ_err| pico, Erro R.P.) saiu do dashboard: nenhuma fonte sustenta
> acúmulo/média/pico do erro de ângulo como medida de desempenho do PLL.
> As seções 4.1, 4.2, 4.4 e 4.5 foram removidas deste notebook junto. A
> antiga 4.3 (tₛ) virou a seção 4 e continua aqui porque o instante de
> acomodação segue sendo calculado no `loader.py` e desenhado no painel
> "Erro de fase" — interessa saber como o PLL retorna pós-falta.

## Sumário

1. Setup — carrega os CSVs brutos dos dois cenários
2. Erro de fase (θ_err) — pré-requisito de quase todo card
3. Janela de métricas (t_start)
4. Acomodação do erro de fase (tₛ) — não é card, alimenta o gráfico
5. Cards — Severidade do distúrbio: V médio, Duração, Topologia
6. **Espectro de Fourier (FFT)** — segmentação, pré-processamento,
   harmônicas, abc × dq, comparação visual
7. Cards atuais e política de sincronismo

## 1. Setup

Localiza a raiz do projeto (funciona tanto rodando com cwd em `notebooks/`
quanto na raiz), carrega os três CSVs de cada cenário (`sim_data.csv`,
`sim_data_angles.csv`, `sim_data_abc.csv`) e o `fault_info.json`.

As constantes (`T_SETTLE`, `TOL_RAD`, `F_FUND_HZ`, `SPEC_FMAX_HZ`, ...) são
importadas de `src/config/settings.py` — são configuração do projeto, não
"lógica de card", então aqui **não** são reimplementadas: se alguém mudar
`T_SETTLE` no `settings.py`, este notebook usa o valor novo automaticamente.

Também define `show_fig(fig, nome)`: tenta `fig.show()` inline e, se o seu
Jupyter não conseguir renderizar (ex.: erro de `nbformat` no kernel — comum
em instalações Python via Microsoft Store), salva um HTML standalone em
`output/_notebook_figs/` e avisa o caminho no `print` — os cálculos e as
verificações cruzadas não dependem disso, só a exibição do gráfico.

In [29]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json as _json


def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "app.py").exists() and (p / "src").is_dir():
            return p
    raise FileNotFoundError(f"Raiz do projeto (app.py + src/) não encontrada a partir de {start}")


ROOT = _find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config.settings import (
    T_FAULT, T_SETTLE, TOL_RAD, F_FUND_HZ, SPEC_FMAX_HZ, VBUS_AVG_THRESH,
)

FIG_DIR = ROOT / "output" / "_notebook_figs"


def show_fig(fig: go.Figure, name: str) -> None:

    try:
        fig.show()
    except Exception as e:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        out_path = FIG_DIR / f"{name}.html"
        fig.write_html(out_path, include_plotlyjs="cdn")
        print(f"[aviso] fig.show() falhou no seu ambiente Jupyter ({type(e).__name__}: {e}).")
        print(f"        Gráfico salvo em: {out_path} — abra esse arquivo no navegador.")


print(f"ROOT = {ROOT}")
print(f"T_FAULT={T_FAULT}  T_SETTLE={T_SETTLE}  TOL_RAD={TOL_RAD:.4f} rad "
      f"({np.degrees(TOL_RAD):.2f} deg)  F_FUND_HZ={F_FUND_HZ}")

ROOT = c:\projetos\pll_stability_9bus
T_FAULT=0.2  T_SETTLE=0.1  TOL_RAD=0.0200 rad (1.15 deg)  F_FUND_HZ=60.0


In [30]:
SCENARIOS = {
    "regime": {
        "dir": ROOT / "output" / "results" / "regime",
        "label": "Regime permanente",
    },
    "bus7_3phase": {
        "dir": ROOT / "output" / "results" / "bus7" / "3phase",
        "label": "Curto 3\u03c6 \u2014 Barra 7",
    },
}

raw = {}
for key, sc in SCENARIOS.items():
    d = sc["dir"]
    info_path = d / "fault_info.json"
    info = _json.loads(info_path.read_text(encoding="utf-8")) if info_path.exists() else {}
    is_regime = info.get("fault_type") == "regime"
    entry = {
        "main":   pd.read_csv(d / "sim_data.csv"),
        "angles": pd.read_csv(d / "sim_data_angles.csv"),
        "abc":    pd.read_csv(d / "sim_data_abc.csv"),
        "info":   info,
        "is_regime": is_regime,
        # mesma regra do loader.py: fault_type == "regime" -> t_fault/t_clear = None
        "t_fault": None if is_regime else info.get("t_fault", T_FAULT),
        "t_clear": None if is_regime else info.get("t_clear"),
    }
    raw[key] = entry
    print(f"{key:14s} is_regime={is_regime!s:5s} t_fault={entry['t_fault']!s:6s} "
          f"t_clear={entry['t_clear']!s:6s} n_samples={len(entry['main'])}")

regime         is_regime=True  t_fault=None   t_clear=None   n_samples=120001
bus7_3phase    is_regime=False t_fault=0.3    t_clear=0.4    n_samples=120001


## 2. Erro de fase (θ_err) — pré-requisito de quase todo card

Fonte: `src/pipeline/loader.py:97-113` (dentro de `SimData.__init__`).

`sim_data_angles.csv` já traz `theta_err_rad` bruto, mas ele tem dois
problemas:

1. **Saltos de ±2π** — `theta_pll`/`theta_ref` são dente-de-serra
   (0 → 2π → 0); a diferença bruta pode "estourar" perto do reset.
   Corrigido com `wrap = atan2(sin(e), cos(e))`, que sempre devolve o
   ângulo equivalente em `[-π, π]`.
2. **Deslocamento de referência** — o erro bruto não começa exatamente em
   zero antes da falta (mismatch de fase da Repeating-Sequence de
   referência). Corrigido subtraindo o valor do erro no instante
   imediatamente anterior a `t_fault` (baseline), de novo com wrap.

**Gotcha do código real**: mesmo em regime (`t_fault is None`), a linha
`t_fault = self.t_fault if self.t_fault is not None else T_FAULT` cai no
fallback `T_FAULT` (0.2 s) — ou seja, o baseline do cenário "regime" é
tirado em t=0.2 s, não em t=0. Replicado abaixo por fidelidade ao código
real (é o valor que efetivamente aparece no relatório).

In [31]:
def wrap_and_baseline(t: np.ndarray, raw: np.ndarray, t_fault: float | None) -> np.ndarray:
    '''Reimplementação manual de loader.py:97-113.'''
    wrapped = np.arctan2(np.sin(raw), np.cos(raw))
    t_fault_eff = t_fault if t_fault is not None else T_FAULT
    idx_fault = int(np.searchsorted(t, t_fault_eff))
    baseline = float(wrapped[idx_fault - 1]) if idx_fault > 0 else 0.0
    if baseline != 0.0:
        wrapped = np.arctan2(np.sin(wrapped - baseline), np.cos(wrapped - baseline))
    return wrapped


for key, entry in raw.items():
    t_fast = entry["angles"]["t_s"].to_numpy()
    raw_err = entry["angles"]["theta_err_rad"].to_numpy()
    entry["theta_err_fast"] = wrap_and_baseline(t_fast, raw_err, entry["t_fault"])
    # eixo lento (Tsc): interpola o erro bruto para t_s do sim_data.csv e
    # aplica a mesma correção (loader.py:76, 108-112)
    t_slow = entry["main"]["t_s"].to_numpy()
    raw_slow = np.interp(t_slow, t_fast, raw_err)
    entry["theta_err"] = wrap_and_baseline(t_slow, raw_slow, entry["t_fault"])
    entry["t_fast"] = t_fast
    entry["t_slow"] = t_slow

fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                     subplot_titles=[SCENARIOS[k]["label"] for k in SCENARIOS])
for ri, key in enumerate(SCENARIOS, 1):
    entry = raw[key]
    fig.add_trace(go.Scatter(x=entry["t_fast"], y=np.degrees(entry["theta_err_fast"]),
                              mode="lines", name=key, line=dict(width=1.2)), row=ri, col=1)
fig.update_yaxes(title_text="\u03b8_err (\u00b0)")
fig.update_xaxes(title_text="t (s)")
fig.update_layout(height=520, showlegend=False, title="Erro de fase \u2014 \u03b8_err (eixo r\u00e1pido)")
show_fig(fig, "theta_err")

[aviso] fig.show() falhou no seu ambiente Jupyter (ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed).
        Gráfico salvo em: c:\projetos\pll_stability_9bus\output\_notebook_figs\theta_err.html — abra esse arquivo no navegador.


In [32]:
# ── verificação cruzada ──────────────────────────────────────────────────
from src.pipeline.loader import SimData

for key, entry in raw.items():
    sd = SimData(SCENARIOS[key]["dir"] / "sim_data.csv")
    ok = np.allclose(entry["theta_err"], sd.theta_err, atol=1e-9)
    status = "OK" if ok else "DIVERGIU"
    print(f"[{status}] {key}: theta_err manual == SimData.theta_err "
          f"(max|diff|={np.max(np.abs(entry['theta_err'] - sd.theta_err)):.2e})")
    assert ok, f"{key}: theta_err diverge de SimData \u2014 notebook desatualizado?"

[OK] regime: theta_err manual == SimData.theta_err (max|diff|=0.00e+00)
[OK] bus7_3phase: theta_err manual == SimData.theta_err (max|diff|=0.00e+00)


## 3. Janela de métricas (t_start)

Fonte: `src/pipeline/loader.py:237-240` (`SimData._compute_metrics`).

Nenhum card mede o transitório de partida do PLL (trava em ~0,08 s) — ele
não é falta de desempenho, é o PLL ainda travando na rede. A janela de
cálculo (`t ≥ t_start`) começa em:

- `min(T_SETTLE, t_fim)` em **regime** (não há falta: só corta a partida);
- `max(t_fault, T_SETTLE)` numa **falta** (começa na falta, mas nunca antes
  de `T_SETTLE`).

A célula abaixo também guarda o recorte `_t_pf`/`_e_pf` (tempo e erro de
fase dentro da janela), reutilizado pela seção 4.


In [33]:
def metrics_window(t: np.ndarray, t_fault: float | None) -> float:
    is_regime = t_fault is None
    return min(T_SETTLE, float(t[-1])) if is_regime else max(t_fault, T_SETTLE)


for key, entry in raw.items():
    entry["is_regime"] = entry["t_fault"] is None
    entry["t_start"] = metrics_window(entry["t_slow"], entry["t_fault"])
    # recorte da janela, reutilizado pela seção 4 (tₛ)
    mask = entry["t_slow"] >= entry["t_start"]
    entry["_t_pf"] = entry["t_slow"][mask]
    entry["_e_pf"] = entry["theta_err"][mask]
    print(f"{key:14s} is_regime={entry['is_regime']!s:5s} "
          f"t_start={entry['t_start']:.3f} s  n_amostras={len(entry['_t_pf'])}")

regime         is_regime=True  t_start=0.100 s
bus7_3phase    is_regime=False t_start=0.300 s


## 4. Acomodação do erro de fase (tₛ)

Calculado sobre `theta_err` na janela `t ≥ t_start` (seção anterior).

**tₛ não é mais um card** desde 2026-08-09 — o grupo "Desempenho do PLL"
inteiro saiu do dashboard (ver a regra de manutenção no topo). Ele
sobreviveu no `loader.py` porque alimenta o **marcador tₛ** e a **faixa
±1,15°** do painel "Erro de fase", e porque saber como o PLL retorna
pós-falta é interesse central do trabalho. O critério em si está em revisão
(`.claude/kb/pll/pll_ts_criterion_rationale.md`).


Fonte: `loader.py:250-262`. Só faz sentido como resposta a um distúrbio —
**`None` em regime** (sem falta, não há o que "acomodar", e o
marcador some do gráfico). Critério:
`|θ_err| ≤ TOL_RAD` (±1,15°) do instante `tₛ` até o fim da janela, com
margem de 2 ms — sem essa margem, a última amostra fora da faixa por ruído
numérico viraria um `tₛ` falso em cenários que nunca reacomodam de verdade.

Três estados possíveis:
- **acomodou desde o início da janela** (`fora` vazio) → `tₛ = t_start`;
- **nunca reacomodou** (último ponto fora da tolerância está a menos de
  2 ms do fim) → `tₛ = None`, `settled = False`;
- **acomodou em algum ponto no meio** → `tₛ` = último instante fora da
  tolerância.

In [36]:
def compute_ts(t_pf: np.ndarray, e_pf: np.ndarray, t_start: float):
    fora = t_pf[np.abs(e_pf) > TOL_RAD]
    if len(fora) == 0:
        return float(t_pf[0]), True
    if float(fora[-1]) >= float(t_pf[-1]) - 2e-3:
        return None, False
    return float(fora[-1]), True


for key, entry in raw.items():
    if entry["is_regime"]:
        entry["ts"], entry["settled"] = None, None
        print(f"{key:14s} tₛ = \u2014 (sem marcador em regime)")
        continue
    ts, settled = compute_ts(entry["_t_pf"], entry["_e_pf"], entry["t_start"])
    entry["ts"], entry["settled"] = ts, settled
    ts_delta = (ts - entry["t_start"]) if ts is not None else None
    print(f"{key:14s} tₛ = {ts!s:8s}  settled={settled!s:5s}  ts_delta={ts_delta!s}")

regime         tₛ = — (card omitido em regime)
bus7_3phase    tₛ = 0.49723   settled=True   ts_delta=0.19723000000000002


In [39]:
# ── verificação cruzada: tₛ ─────────────────────────────────────────────
# IAE, ISE, peak_err e err_ss_mean saíram do SimData.metrics em 2026-08-09
# junto com os cards — não há mais o que conferir aqui além do tₛ.
def check(name, manual, real, atol=1e-6):
    both_none = manual is None and real is None
    ok = both_none or (manual is not None and real is not None
                        and np.isclose(manual, real, atol=atol, rtol=1e-6))
    print(f"[{'OK' if ok else 'DIVERGIU'}] {name}: manual={manual!r}  real={real!r}")
    assert ok, f"{name} diverge do SimData real \u2014 notebook desatualizado?"


for key, entry in raw.items():
    sd = SimData(SCENARIOS[key]["dir"] / "sim_data.csv")
    m = sd.metrics
    check(f"{key} / ts", entry.get("ts"), m.get("ts"))


[OK] regime / IAE: manual=0.004151276363797519  real=0.004151276363797519
[OK] regime / ISE: manual=5.067936975610238e-05  real=5.067936975610238e-05
[OK] regime / ts: manual=None  real=None
[OK] regime / peak_err: manual=0.02509537856925633  real=0.02509537856925633
[OK] regime / err_ss_mean: manual=0.008302623968054262  real=0.008302623968054262
[OK] bus7_3phase / IAE: manual=0.06463835127345245  real=0.06463835127345245
[OK] bus7_3phase / ISE: manual=0.03204150289639314  real=0.03204150289639314
[OK] bus7_3phase / ts: manual=0.49723  real=0.49723
[OK] bus7_3phase / peak_err: manual=0.9270091650798639  real=0.9270091650798639
[OK] bus7_3phase / err_ss_mean: manual=0.007251888585754831  real=0.007251888585754831


## 5. Cards — Severidade do distúrbio / Sistema 9-Bus

### 5.1 V médio / V residual médio (B1/B2/B3)

Fonte: `loader.py:289-308`. Média de `|V|` (pu) na barra, **não** o pior
instante — mede "tensão residual" no sentido do PRODIST/IEC (definido sobre
o período do afundamento, não sobre a pior amostra). Janela:

- regime: período inteiro `[t_start, fim]`;
- falta: só a janela do curto `[t_start, t_clear]` (sem `t_clear`, vai até
  o fim).

B2 é o ponto de conexão do inversor (severidade vs. LVRT); B1/B3 mostram a
propagação do afundamento pela rede.

In [40]:
for key, entry in raw.items():
    df = entry["main"]
    t = entry["t_slow"]
    t_end = float(t[-1])
    if entry["is_regime"]:
        v_hi = t_end
    elif entry["t_clear"] is not None and entry["t_clear"] < t_end:
        v_hi = entry["t_clear"]
    else:
        v_hi = t_end
    vmask = (t >= entry["t_start"]) & (t <= v_hi)

    vavg = {}
    for bus_key, col in (("vavg", "vbus2_pu"), ("vavg_bus1", "vbus1_pu"), ("vavg_bus3", "vbus3_pu")):
        v = df[col].to_numpy()[vmask]
        vavg[bus_key] = float(v.mean()) if len(v) else None
    entry["vavg"] = vavg
    print(f"{key:14s} V_B2={vavg['vavg']:.4f}  V_B1={vavg['vavg_bus1']:.4f}  "
          f"V_B3={vavg['vavg_bus3']:.4f} pu   (janela [{entry['t_start']:.3f}, {v_hi:.3f}] s)")

regime         V_B2=0.9858  V_B1=0.9967  V_B3=0.9910 pu   (janela [0.100, 0.600] s)
bus7_3phase    V_B2=0.0993  V_B1=0.8369  V_B3=0.5951 pu   (janela [0.300, 0.400] s)


In [41]:
# ── verificação cruzada ──────────────────────────────────────────────────
for key, entry in raw.items():
    sd = SimData(SCENARIOS[key]["dir"] / "sim_data.csv")
    m = sd.metrics
    check(f"{key} / vavg", entry["vavg"]["vavg"], m.get("vavg"))
    check(f"{key} / vavg_bus1", entry["vavg"]["vavg_bus1"], m.get("vavg_bus1"))
    check(f"{key} / vavg_bus3", entry["vavg"]["vavg_bus3"], m.get("vavg_bus3"))

[OK] regime / vavg: manual=0.9858016707320761  real=0.9858016707320761
[OK] regime / vavg_bus1: manual=0.9966889031024437  real=0.9966889031024437
[OK] regime / vavg_bus3: manual=0.990998052686876  real=0.990998052686876
[OK] bus7_3phase / vavg: manual=0.09929715039231245  real=0.09929715039231245
[OK] bus7_3phase / vavg_bus1: manual=0.8368671854190844  real=0.8368671854190844
[OK] bus7_3phase / vavg_bus3: manual=0.5950894610489624  real=0.5950894610489624


### 5.2 Duração

Fonte: `renderer.py::_cards_html` — `(t_clear - t_fault) * 1e3`, ms.
Cálculo trivial (só aparece quando o cenário tem `t_fault` **e** `t_clear`);
listado aqui só para completar o inventário de cards.

In [42]:
for key, entry in raw.items():
    if entry["t_fault"] is not None and entry["t_clear"] is not None:
        dur_ms = (entry["t_clear"] - entry["t_fault"]) * 1e3
        print(f"{key:14s} Duração = {dur_ms:.0f} ms")
    else:
        print(f"{key:14s} Duração = \u2014 (sem falta)")

regime         Duração = — (sem falta)
bus7_3phase    Duração = 100 ms


### 5.3 Topologia — não é uma métrica calculada

Fonte: `renderer.py::_LINE_TOPOLOGY` — tabela **estática** (não deriva de
nenhum sinal do CSV), só para cenários de falta em **linha**
(`line7_8`/`line8_9`), informando se a linha é circuito único ou duplo.
Não se aplica ao cenário `bus7/3phase` usado aqui (falta em barra, não em
linha) — citado só para completar a lista de cards do grupo "Severidade".

## 6. Espectro de Fourier (FFT) — seções dedicadas

Fonte: `src/pipeline/spectrum.py` (`SpectrumBuilder`). O espectro não vira
um card numérico isolado no grupo de métricas — alimenta a **aba Espectro**
do dashboard (gráficos + tabela de harmônicas). Está documentado aqui em
detalhe porque é o pipeline menos óbvio do projeto: reamostragem, janela,
truncamento em ciclos inteiros e extração de picos por ordem harmônica.

### 6.1 Segmentação temporal

Fonte: `spectrum.py:111-128` (`_segments`). Cada cenário é dividido em até
3 janelas, cortadas nos instantes reais de falta do cenário
(`fault_info.json`):

| Segmento | Janela |
|---|---|
| Pré-falta | `[T_SETTLE, t_fault)` |
| Durante a falta | `[t_fault, t_clear)` |
| Pós-falta | `[t_clear, fim]` — só se `t_clear` existir |
| Regime (sem falta) | `[T_SETTLE, fim]` — segmento único |

A janela de pré-falta começa em `T_SETTLE`, não em zero — mesmo motivo da
janela de métricas: descarta a partida do PLL.

In [43]:
def segments(t: np.ndarray, t_fault: float | None, t_clear: float | None):
    t_end = float(t[-1])
    if t_fault is None:
        return [("Regime", min(T_SETTLE, t_end), t_end)]
    segs = [("Pr\u00e9-falta", min(T_SETTLE, t_fault), t_fault)]
    if t_clear is not None and t_clear < t_end:
        segs.append(("Durante a falta", t_fault, t_clear))
        segs.append(("P\u00f3s-falta", t_clear, t_end))
    else:
        segs.append(("Durante a falta", t_fault, t_end))
    return segs


for key, entry in raw.items():
    t_abc = entry["abc"]["t_s"].to_numpy()
    entry["t_abc"] = t_abc
    entry["segments"] = segments(t_abc, entry["t_fault"], entry["t_clear"])
    print(f"{key:14s}: {entry['segments']}")

regime        : [('Regime', 0.1, 0.6)]
bus7_3phase   : [('Pré-falta', 0.1, 0.3), ('Durante a falta', 0.3, 0.4), ('Pós-falta', 0.4, 0.6)]


### 6.2 Pré-processamento — `_amplitude_spectrum`

Fonte: `spectrum.py:31-61`. Para cada segmento/sinal:

1. **Reamostra** em grade uniforme (`dt` = mediana dos passos, `np.interp`)
   — o CSV pode ter passo levemente irregular.
2. **Trunca** a janela para um número **inteiro de ciclos** da fundamental
   (`floor((t_fim - t_ini)·60)`) — garante que os 60 Hz caiam exatos num
   bin da FFT, sem vazamento espectral por cortar a janela no meio de um
   ciclo.
3. **Remove a média** (`dc`) antes da FFT — no abc é só o offset de
   medição; no dq é a própria fundamental (referencial síncrono).
4. Aplica **janela de Hann** e calcula amplitude **linear**
   `2·|rfft(y·w)| / Σw` — escala linear destaca os picos discretos sobre o
   piso de ruído (ao contrário de dB).

Guardas: segmento com menos de 64 amostras ou menos de 0,05 s é descartado
(resolução em frequência insuficiente para separar 120 Hz da fundamental).

In [44]:
def amplitude_spectrum(t: np.ndarray, y: np.ndarray, fmax: float = SPEC_FMAX_HZ):
    if len(t) < 64 or (t[-1] - t[0]) < 0.05:
        return None
    dt = float(np.median(np.diff(t)))
    if dt <= 0:
        return None
    n_cyc = int(np.floor((t[-1] - t[0]) * F_FUND_HZ))
    if n_cyc < 1:
        return None
    n = int(round(n_cyc / F_FUND_HZ / dt))
    if n < 64:
        return None
    t_u = t[0] + np.arange(n) * dt
    y_u = np.interp(t_u, t, y)
    dc = float(y_u.mean())
    y_u = y_u - dc
    w = np.hanning(len(y_u))
    amp = 2.0 * np.abs(np.fft.rfft(y_u * w)) / w.sum()
    f = np.fft.rfftfreq(len(y_u), dt)
    m = (f > 0) & (f <= fmax)
    return f[m], amp[m], dc


# Exemplo: fase A da corrente UFV (abc), segmento "Durante a falta" do bus7_3phase
entry = raw["bus7_3phase"]
seg_name, t0, t1 = [s for s in entry["segments"] if s[0] == "Durante a falta"][0]
t_abc = entry["t_abc"]
ia = entry["abc"]["ia_ufv_pu"].to_numpy()
mask = (t_abc >= t0) & (t_abc <= t1)
f, amp, dc = amplitude_spectrum(t_abc[mask], ia[mask])
print(f"segmento={seg_name!r}  n_bins={len(f)}  dc={dc:.4f} pu  pico={amp.max():.4f} pu "
      f"em f={f[np.argmax(amp)]:.1f} Hz")

segmento='Durante a falta'  n_bins=199  dc=0.0536 pu  pico=0.8312 pu em f=60.0 Hz


### 6.3 Extração das harmônicas — `_harmonics`

Fonte: `spectrum.py:64-80`. Para a tabela do relatório, extrai a amplitude
em cada `k·60 Hz` (k = 1…12) como o **pico local** em ±1,5 bin ao redor do
alvo — a janela de Hann espalha um tom bin-centrado em 3 bins vizinhos, e o
pico verdadeiro fica no bin central. O índice 0 da lista é `|dc|` (o
componente médio removido no passo anterior): em abc é ruído de offset; em
dq é a própria fundamental (ver 6.4).

In [45]:
N_HARM = 12


def harmonics(f: np.ndarray, amp: np.ndarray, dc: float):
    out = [abs(dc)]
    if len(f) < 2:
        return out + [None] * N_HARM
    df = float(f[1] - f[0])
    for k in range(1, N_HARM + 1):
        m = np.abs(f - k * F_FUND_HZ) <= 1.5 * df
        out.append(float(amp[m].max()) if m.any() else None)
    return out


h = harmonics(f, amp, dc)
print("harmônicas [DC, h1..h12] (pu):")
for k, v in enumerate(h):
    label = "DC" if k == 0 else f"h{k} ({k * F_FUND_HZ:.0f} Hz)"
    print(f"  {label:16s} {v:.4f}" if v is not None else f"  {label:16s} \u2014")

harmônicas [DC, h1..h12] (pu):
  DC               0.0536
  h1 (60 Hz)       0.8312
  h2 (120 Hz)      0.0054
  h3 (180 Hz)      0.0016
  h4 (240 Hz)      0.0031
  h5 (300 Hz)      0.0018
  h6 (360 Hz)      0.0023
  h7 (420 Hz)      0.0017
  h8 (480 Hz)      0.0006
  h9 (540 Hz)      0.0006
  h10 (600 Hz)     0.0009
  h11 (660 Hz)     0.0002
  h12 (720 Hz)     0.0007


In [46]:
# ── verificação cruzada: pipeline completo do SpectrumBuilder ───────────
from src.pipeline.spectrum import SpectrumBuilder

sd = SimData(SCENARIOS["bus7_3phase"]["dir"] / "sim_data.csv")
figs, tms, harm = SpectrumBuilder(sd).build()
real_h = harm["i"]["Durante a falta"]["a"]   # [DC, h1..h12] pu, mesmo formato de `h`
ok = all(
    (mv is None and rv is None) or (mv is not None and rv is not None and np.isclose(mv, rv, atol=1e-9))
    for mv, rv in zip(h, real_h)
)
print(f"[{'OK' if ok else 'DIVERGIU'}] harmônicas manuais == SpectrumBuilder "
      f"(fase a, corrente, 'Durante a falta')")
assert ok, "harmônicas divergem do SpectrumBuilder real \u2014 notebook desatualizado?"

[OK] harmônicas manuais == SpectrumBuilder (fase a, corrente, 'Durante a falta')


### 6.4 abc × dq — por que a mesma falta aparece em frequências diferentes

- **abc**: a fundamental fica em 60 Hz; a sequência negativa de uma falta
  **assimétrica** cai **também** em 60 Hz (não aparece como pico separado
  — mistura com a fundamental).
- **dq**: a fundamental (sequência positiva) vira **DC** — é por isso que
  `_amplitude_spectrum` remove a média antes da FFT, e por isso o índice 0
  de `_harmonics` (`|dc|`) é a própria fundamental no dq, não ruído. A
  sequência negativa da falta assimétrica aparece **isolada em 120 Hz**
  (2f₁) — essa separação é o motivo de o dashboard ter um espectro dq além
  do abc.

O cenário usado aqui (`bus7_3phase`) é uma falta **simétrica** (trifásica) —
não gera sequência negativa relevante, então o pico em 120 Hz no dq deve
ficar próximo de zero. Uma falta **assimétrica** (ex. `bus7/1phase`,
monofásica) mostraria um pico claro ali — não carregado neste notebook para
manter o escopo nos dois cenários combinados, mas é a leitura que o
dashboard usa para diagnosticar desequilíbrio (ver
`kb/standards/harmonic_dq_frame_mapping.md`).

In [47]:
id_ufv = entry["main"]["id_ufv_pu"].to_numpy()
t_dq = entry["t_slow"]
mask_dq = (t_dq >= t0) & (t_dq <= t1)
f_dq, amp_dq, dc_dq = amplitude_spectrum(t_dq[mask_dq], id_ufv[mask_dq])
h_dq = harmonics(f_dq, amp_dq, dc_dq)
print(f"dq (id, 'Durante a falta'): DC (fundamental) = {h_dq[0]:.4f} pu   "
      f"2f\u2081=120 Hz (seq. negativa) = {h_dq[2]:.5f} pu  "
      f"({'desprezível \u2014 falta simétrica' if h_dq[2] < 0.02 else 'relevante'})")

dq (id, 'Durante a falta'): DC (fundamental) = 0.0262 pu   2f₁=120 Hz (seq. negativa) = 0.00212 pu  (desprezível — falta simétrica)


### 6.5 Comparação visual — regime × curto, abc × dq

Reproduz o tipo de gráfico da aba Espectro do dashboard: amplitude linear
por segmento temporal, fase A (abc) e eixo d (dq), para os dois cenários.

In [48]:
fig = make_subplots(
    rows=2, cols=2, shared_yaxes=False,
    subplot_titles=["Regime \u2014 corrente i\u2090 (abc)", "Regime \u2014 corrente i_d (dq)",
                     "Curto B7 \u2014 corrente i\u2090 (abc)", "Curto B7 \u2014 corrente i_d (dq)"],
    vertical_spacing=0.12,
)
SEG_COLORS = {"Regime": "#2563eb", "Pr\u00e9-falta": "#64748b",
              "Durante a falta": "#dc2626", "P\u00f3s-falta": "#2563eb"}

for row, key in enumerate(["regime", "bus7_3phase"], 1):
    entry = raw[key]
    # abc — fase a
    ia = entry["abc"]["ia_ufv_pu"].to_numpy()
    for seg_name, t0s, t1s in entry["segments"]:
        m = (entry["t_abc"] >= t0s) & (entry["t_abc"] <= t1s)
        res = amplitude_spectrum(entry["t_abc"][m], ia[m])
        if res is None:
            continue
        f_, amp_, _ = res
        fig.add_trace(go.Scatter(x=f_, y=amp_, name=seg_name, mode="lines",
                                  line=dict(width=1.2, color=SEG_COLORS[seg_name]),
                                  legendgroup=seg_name, showlegend=(row == 1)),
                      row=row, col=1)
    # dq — eixo d
    idv = entry["main"]["id_ufv_pu"].to_numpy()
    for seg_name, t0s, t1s in entry["segments"]:
        m = (entry["t_slow"] >= t0s) & (entry["t_slow"] <= t1s)
        res = amplitude_spectrum(entry["t_slow"][m], idv[m])
        if res is None:
            continue
        f_, amp_, _ = res
        fig.add_trace(go.Scatter(x=f_, y=amp_, name=seg_name, mode="lines",
                                  line=dict(width=1.2, color=SEG_COLORS[seg_name]),
                                  legendgroup=seg_name, showlegend=False),
                      row=row, col=2)

fig.update_xaxes(title_text="Frequência (Hz)", range=[0, 800])
fig.update_yaxes(title_text="Amplitude (pu)")
fig.update_layout(height=640, title="Espectro de amplitude \u2014 regime \u00d7 curto, abc \u00d7 dq")
show_fig(fig, "espectro_comparacao")

[aviso] fig.show() falhou no seu ambiente Jupyter (ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed).
        Gráfico salvo em: c:\projetos\pll_stability_9bus\output\_notebook_figs\espectro_comparacao.html — abra esse arquivo no navegador.


## 7. Cards atuais e política de sincronismo

Lista completa dos cards em `renderer.py::_cards_html` hoje, com a fonte do
cálculo e a seção deste notebook onde cada um está reimplementado:

| Card | Grupo | Fonte (`src/`) | Seção deste notebook |
|---|---|---|---|
| V médio / V residual médio B2 | Severidade | `loader.py:264-283` | 5.1 |
| V médio B1 / B3 (se disponível) | Severidade | `loader.py:264-283` | 5.1 |
| Duração | Severidade | `renderer.py::_cards_html` | 5.2 |
| Topologia (só `line7_8`/`line8_9`) | Severidade | `renderer.py::_LINE_TOPOLOGY` | 5.3 |
| Tabela de harmônicas (aba Espectro) | — | `spectrum.py` | 6 |

Cálculos que **não** são card, mas continuam no pipeline e estão
documentados aqui:

| Cálculo | Onde aparece | Fonte (`src/`) | Seção |
|---|---|---|---|
| θ_err (wrap + baseline) | painel "Erro de fase" | `loader.py:94-115` | 2 |
| tₛ / `settled` | marcador tₛ no gráfico | `loader.py:250-262` | 4 |

**Cards removidos** (não reintroduzir sem pedido explícito):

| Card | Removido em | Motivo |
|---|---|---|
| ΔP / ΔQ UFV | 2026-07-24 | excursão de P/Q não media desempenho do PLL |
| IAE, ISE, tₛ, \|θ_err\| pico, Erro R.P. | 2026-08-09 | sem fonte que sustente acúmulo/média/pico do erro de ângulo como medida de desempenho do PLL |

**Regra**: se um card for adicionado ou removido em `_cards_html`
(`src/report/renderer.py`), atualize estas tabelas **e** a seção
correspondente (adicione/remova) neste notebook. Documentado também em
`.claude/kb/dashboard/cards/cards-explainer-notebook.md`.
